# Caliby

![Caliby](https://proto-bio.github.io/proto-assets/images/tool/caliby/hero.png)

Caliby is a structure-conditioned protein design model that predicts the parameters of a Potts
model from backbone geometry and samples sequences by MCMC under the resulting energy, rather
than decoding residues autoregressively. Because the Potts parameters can be aggregated
position-wise over several backbones, one design run can be conditioned on a whole conformer
ensemble instead of a single static structure.

- Paper: [doi.org/10.1101/2025.09.30.679633](https://doi.org/10.1101/2025.09.30.679633)
- Code: [ProteinDesignLab/caliby](https://github.com/ProteinDesignLab/caliby)

This notebook demonstrates three functions. `run_caliby_sample` designs sequences for a target
backbone, `run_caliby_pack` places sidechains onto a backbone that already carries a sequence,
and `run_caliby_score` reports the Potts energy of the sequence a structure already carries.

In [1]:
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools, display_doc_link

display_doc_link("caliby")

## Available tools

In [2]:
display_available_tools("caliby")

- **`run_caliby_pack()`** — Pack protein sidechains onto a backbone using Caliby's diffusion packer
- **`run_caliby_sample()`** — Design protein sequences with Caliby, optionally conditioned on a conformer ensemble
- **`run_caliby_score()`** — Score a structure's native sequence under the Caliby Potts model

### `run_caliby_sample`

Caliby reads a backbone and predicts the fields and couplings of a Potts model over residue
identities, then anneals an MCMC sampler from temperature 1.0 down to `temperature` to draw
sequences from that energy landscape. The couplings make the sampler aware of pairwise residue
compatibility across the whole chain at once, which is what distinguishes it from an
autoregressive decoder.

Each design reports its Potts energy, where lower is more favorable, together with the fraction
of designed positions that recover the native residue. Designs are backbone-only: the residue
identities are threaded onto the input coordinates and no sidechain atoms are generated. The
`caliby-pack` section below places them.

In [3]:
from pathlib import Path

from proto_tools.entities.structures import Structure
from proto_tools.tools.inverse_folding.caliby import (
    CalibySampleConfig,
    CalibySampleInput,
    CalibyStructureInput,
    run_caliby_sample,
)

# Resolve the in-tree 65-residue single-chain fixture by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
backbone = Structure.from_file(
    str(_repo_root / "proto_tools" / "tools" / "inverse_folding" / "example_input_fixture.pdb")
)
print(f"Loaded chains {backbone.get_chain_ids()} with {len(backbone.get_chain_sequence('A'))} residues")

Loaded chains ['A'] with 65 residues


In [4]:
# Display input docs
display_api_reference("caliby", "input", "run_caliby_sample")

# Redesign chain A while holding a few positions at their native identity
sample_input = CalibySampleInput(
    inputs=[
        CalibyStructureInput(
            structure=backbone,
            chains_to_redesign=["A"],
            fixed_positions={"A": [10, 15, 20]},
        )
    ]
)

**Input** — `CalibySampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[CalibyStructureInput]</code> | required | Per-structure inputs, each a primary conformer plus optional conformers. |

In [5]:
# Display config docs
display_api_reference("caliby", "config", "run_caliby_sample")

# Configure sampling | see docs above for all fields
sample_config = CalibySampleConfig(
    num_sequences_per_structure=2,
    temperature=0.1,  # Final annealed Potts temperature; the default 0.01 is near-greedy
    potts_sweeps=200,  # Use 500 or more for production runs
    excluded_amino_acids=["C"],  # Keep cysteine out of the designs
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `CalibySampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on, e.g. 'cuda', 'cuda:0' or 'cpu'. |
| <code>model_checkpoint</code> | <code>str</code> | <code>'caliby'</code> | Bundled design checkpoint name, or a path to a custom .ckpt file. |
| <code>num_sequences_per_structure</code> | <code>int</code> | <code>1</code> | Number of sequences to design per input structure. |
| <code>batch_size</code> | <code>int</code> | <code>4</code> | Structure instances per forward pass; reduce on GPU out-of-memory errors. |
| <code>temperature</code> | <code>float</code> | <code>0.01</code> | Final Potts sampling temperature, annealed from 1.0; higher = more diverse. |
| <code>potts_sweeps</code> | <code>int</code> | <code>500</code> | MCMC sweeps per designed sequence; more converges closer to the optimum. |
| <code>excluded_amino_acids</code> | <code>list[Literal['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']] &#124; None</code> | <code>None</code> | One-letter codes never placed at any designed position. |
| <code>use_primary_res_type</code> | <code>bool</code> | <code>True</code> | Take residue identities from the primary conformer (ensemble design only). |

In [6]:
# Run sequence design
sample_result = run_caliby_sample(sample_input, sample_config)

Running run_caliby_sample [00:00]

In [7]:
# Display output docs
display_api_reference("caliby", "output", "run_caliby_sample")

# Print each designed sequence with its Potts energy and native-sequence recovery
for i, design_set in enumerate(sample_result.design_sets):
    for j, design in enumerate(design_set.complexes):
        sequence = design.chains[0].sequence
        shown = f"{sequence[:60]}..." if len(sequence) > 60 else sequence
        print(f"Structure {i}, design {j + 1}: {shown}")
        print(f"  Potts energy: {design.metrics['potts_energy']:.2f} (lower is better)")
        print(f"  Per-position energy: {design.metrics['avg_potts_energy']:.3f}")
        print(f"  Sequence recovery: {design.metrics['sequence_recovery']:.1%}")

**Output** — `CalibySampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>design_sets</code> | <code>list[CalibyDesignSet]</code> | required | One CalibyDesignSet per input structure, in input order. |

**`CalibyDesignSet`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[CalibyDesign]</code> | required | Caliby complexes designed for one input structure. |

**`CalibyDesign`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chains</code> | <code>list[Chain &#124; Fragment]</code> | required | Chains in the complex. Strings, dicts, Chain, Fragment, or Ligands. |
| <code>designed</code> | <code>list[bool]</code> | required | Per-chain redesign status, aligned by position with chains. |
| <code>metrics</code> | <code>CalibyDesignMetrics</code> | <code>CalibyDesignMetrics(primary_metric='potts_energy', metric_type='CalibyDesignMetrics')</code> | Per-design Potts energy and native-sequence recovery. |
| <code>structure</code> | <code>Structure</code> | required | Input backbone with designed residue identities; sidechains not packed. |

**Metrics** — `CalibyDesignMetrics` (one set per `design_sets` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>potts_energy</code> **(primary)** | <code>float</code> |  |  |  |
| <code>avg_potts_energy</code> | <code>float</code> |  |  |  |
| <code>sequence_recovery</code> | <code>float</code> | <code>[0, 1]</code> |  |  |

Structure 0, design 1: GWEELDLKAFVEKHNLERVARELGLSVETVKEILKKNYKVYVRRDADGRLHVYVEKDGKW...
  Potts energy: -156.13 (lower is better)
  Per-position energy: -2.402
  Sequence recovery: 44.6%
Structure 0, design 2: MKKKLDLKAFVEKYNIEKVAKELGLSKRTVEEILKNDYEVYVETTPSGKKIVYVKKDGKW...
  Potts energy: -157.82 (lower is better)
  Per-position energy: -2.428
  Sequence recovery: 49.2%


### `run_caliby_pack`

Sequence design leaves the backbone without sidechains, so full-atom analysis needs a packing
step. `run_caliby_pack` places sidechain atoms for the residue identities a structure already
carries, by per-token Euclidean diffusion. Packing uses a separate `caliby_packer_*` checkpoint
rather than a design checkpoint; the numeric suffix is the training noise level in Angstroms, so
`caliby_packer_000` suits noise-free experimental backbones, `caliby_packer_010` is the general
recommendation, and `caliby_packer_030` tolerates noisier coordinates.

In [8]:
from proto_tools.tools.inverse_folding.caliby import (
    CalibyPackConfig,
    CalibyPackInput,
    run_caliby_pack,
)

In [9]:
# Display input docs
display_api_reference("caliby", "input", "run_caliby_pack")

# Pack sidechains onto the native backbone
pack_input = CalibyPackInput(structures=[backbone])

**Input** — `CalibyPackInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | Structures to pack sidechains onto, each carrying the sequence to pack. |

In [10]:
# Display config docs
display_api_reference("caliby", "config", "run_caliby_pack")

# Configure packing | see docs above for all fields
pack_config = CalibyPackConfig(
    model_checkpoint="caliby_packer_010",  # 0.1 A training noise, recommended default
    diffusion_steps=50,
    step_scale=1.5,
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `CalibyPackConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on, e.g. 'cuda', 'cuda:0' or 'cpu'. |
| <code>model_checkpoint</code> | <code>str</code> | <code>'caliby_packer_010'</code> | Bundled packer checkpoint name, or a path to a custom .ckpt file. |
| <code>batch_size</code> | <code>int</code> | <code>4</code> | Structures packed per forward pass; reduce on GPU out-of-memory errors. |
| <code>diffusion_steps</code> | <code>int</code> | <code>50</code> | Number of sidechain diffusion denoising steps. |
| <code>step_scale</code> | <code>float</code> | <code>1.5</code> | Step scale (eta) for sidechain diffusion. |

In [11]:
# Run sidechain packing
pack_result = run_caliby_pack(pack_input, pack_config)

Running run_caliby_pack [00:00]

In [12]:
# Display output docs
display_api_reference("caliby", "output", "run_caliby_pack")


# Count atoms in the first model of a structure
def atom_count(structure: Structure) -> int:
    return sum(len(residue) for chain in structure.gemmi_struct[0] for residue in chain)


# Compare atom counts before and after packing
print(f"Input structure: {atom_count(backbone)} atoms over {backbone.num_residues} residues")
for i, packed in enumerate(pack_result.packed_structures):
    print(f"Packed structure {i}: {atom_count(packed)} atoms")

**Output** — `CalibyPackOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>packed_structures</code> | <code>list[Structure]</code> | required | Packed full-atom structures, one per input structure, in input order. |

Input structure: 526 atoms over 65 residues
Packed structure 0: 591 atoms


### `run_caliby_score`

`run_caliby_score` computes Potts parameters for a backbone and evaluates the energy of the
sequence that structure already carries, so there is no separate sequence input: to score a
variant, thread it onto the backbone first. The reported `potts_energy` is an unnormalized Potts
energy where **lower is better**, not a log-likelihood or a perplexity. Probability is
proportional to `exp(-U/T)` for a sampling temperature that scoring never fixes, which is why no
perplexity is reported. The energy is a sum over positions and therefore grows with sequence
length, so `avg_potts_energy` is the value to compare across structures of different sizes.

Setting `return_local_energies` additionally returns the per-position local conditional energy
matrix, with `potts_tokens` giving its column order.

In [13]:
from proto_tools.tools.inverse_folding.caliby import (
    CalibyScoreConfig,
    CalibyScoreInput,
    CalibyScoreStructureInput,
    run_caliby_score,
)

In [14]:
# Display input docs
display_api_reference("caliby", "input", "run_caliby_score")

# Score the native sequence the backbone already carries
score_input = CalibyScoreInput(inputs=[CalibyScoreStructureInput(structure=backbone)])

**Input** — `CalibyScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[CalibyScoreStructureInput]</code> | required | Per-structure inputs, each scored on its own native sequence. |

In [15]:
# Display config docs
display_api_reference("caliby", "config", "run_caliby_score")

# Configure scoring | see docs above for all fields
score_config = CalibyScoreConfig(
    return_local_energies=True,  # Also return the per-position energy matrix
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `CalibyScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on, e.g. 'cuda', 'cuda:0' or 'cpu'. |
| <code>model_checkpoint</code> | <code>str</code> | <code>'caliby'</code> | Bundled design checkpoint name, or a path to a custom .ckpt file. |
| <code>batch_size</code> | <code>int</code> | <code>4</code> | Structure instances per forward pass; reduce on GPU out-of-memory errors. |
| <code>return_local_energies</code> | <code>bool</code> | <code>False</code> | Include the per-position local conditional energy matrix in the output. |

In [16]:
# Run Potts-energy scoring
score_result = run_caliby_score(score_input, score_config)

Running run_caliby_score [00:00]

In [17]:
# Display output docs
display_api_reference("caliby", "output", "run_caliby_score")

# Print the scored sequence and its Potts energy (lower is better)
for i, score in enumerate(score_result.scores):
    shown = f"{score.sequence[:60]}..." if len(score.sequence) > 60 else score.sequence
    print(f"Structure {i}: {shown}")
    print(f"  Potts energy: {score['potts_energy']:.2f} (lower is better)")
    print(f"  Per-position energy: {score['avg_potts_energy']:.3f}")
    if score.local_conditional_energies is not None:
        rows = len(score.local_conditional_energies)
        cols = len(score.local_conditional_energies[0])
        print(f"  Local energy matrix: {rows} positions x {cols} tokens")
        print(f"  Token order: {''.join(score_result.potts_tokens or [])}")

**Output** — `CalibyScoreOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[CalibyScoringMetrics]</code> | required | Potts-energy scores, one per input structure, in input order. |

**Metrics** — `CalibyScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>potts_energy</code> **(primary)** | <code>float</code> |  |  |  |
| <code>avg_potts_energy</code> | <code>float</code> |  |  |  |

Structure 0: MRKKLDLKKFVEDKNQEYAARALGLSQKLIEEVLKRGLPVYVETNKDGNIKVYITQDGIT...
  Potts energy: -113.77 (lower is better)
  Per-position energy: -1.750
  Local energy matrix: 65 positions x 22 tokens
  Token order: ALAARGASNASPCYSGLNGLUGLYHISILELEULYSMETPHEPROSERTHRTRPTYRVALUNK<G>


### Ensemble-conditioned design

Passing `conformers` switches Caliby from single-structure design to multistate design: the Potts
parameters are aggregated position-wise over `[structure, *conformers]`, so the sampled sequence
has to be compatible with every backbone in the set at once. The inherited `structure` field is
the **primary conformer**, meaning its residue identities are read as the native sequence and
broadcast across the ensemble when `use_primary_res_type` is set, and it is the reference frame
the other members' chain IDs and residue counts must match.

A real ensemble comes from molecular dynamics or from a generative sampler. The
`StructureEnsemble.structures` list returned by `bioemu-sample` can be passed straight into
`conformers`. The cell below repeats the single fixture a couple of times so the example stays
runnable without an MD trajectory; the energies it produces are therefore the single-structure
ones, and only the shape of the call is illustrative.

In [18]:
# Build a (stand-in) ensemble: primary conformer first, other members in `conformers`
ensemble_input = CalibySampleInput(
    inputs=[
        CalibyStructureInput(
            structure=backbone,
            conformers=[backbone, backbone],  # Replace with e.g. ensemble.structures from bioemu-sample
            chains_to_redesign=["A"],
        )
    ]
)

ensemble_config = CalibySampleConfig(
    num_sequences_per_structure=1,
    temperature=0.1,
    potts_sweeps=200,
    use_primary_res_type=True,  # Read residue identities from the primary conformer
    batch_size=3,  # How many conformers pass through the model at once
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

ensemble_result = run_caliby_sample(ensemble_input, ensemble_config)

ensemble_design = ensemble_result.design_sets[0].complexes[0]
print(f"Designed against {1 + len(ensemble_input.inputs[0].conformers)} conformers")
print(f"Potts energy: {ensemble_design.metrics['potts_energy']:.2f} (lower is better)")
print(f"Sequence recovery: {ensemble_design.metrics['sequence_recovery']:.1%}")

Running run_caliby_sample [00:00]

Designed against 3 conformers
Potts energy: -158.95 (lower is better)
Sequence recovery: 32.3%


### From design to full-atom model

`run_caliby_sample` returns backbone-only designs: the designed residue identities sit on the
input coordinates with no sidechain atoms. Feeding `design.structure` into `run_caliby_pack`
completes the model, which is what downstream steps such as structure relaxation, interface
scoring, or ligand docking need.

In [19]:
# Pack sidechains onto every design from the sampling run
designs = [design.structure for design_set in sample_result.design_sets for design in design_set.complexes]

packed_designs = run_caliby_pack(CalibyPackInput(structures=designs), pack_config)

for i, (design_backbone, packed) in enumerate(zip(designs, packed_designs.packed_structures, strict=True)):
    print(f"Design {i + 1}: {atom_count(design_backbone)} backbone atoms -> {atom_count(packed)} full-atom atoms")

Running run_caliby_pack [00:00]

Running run_caliby_pack [00:00]

Design 1: 260 backbone atoms -> 614 full-atom atoms
Design 2: 260 backbone atoms -> 609 full-atom atoms


## Export Results

Designed sequences export to FASTA or JSON, packed structures to CIF or PDB, and Potts-energy
scores to CSV or JSON.

In [20]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="caliby_designs", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'caliby_designs/'}")

packed_designs.export(name="caliby_packed_designs", export_path=output_dir, file_format="pdb")
print(f"Packed designs exported to {output_dir / 'caliby_packed_designs/'}")

score_result.export(name="caliby_scores", export_path=output_dir, file_format="csv")
print(f"Potts energies exported to {output_dir / 'caliby_scores.csv'}")

Designed sequences exported to example_output/caliby_designs
Packed designs exported to example_output/caliby_packed_designs
Potts energies exported to example_output/caliby_scores.csv
